# Field Strength Operators Walkthrough

## Setup
### Imports & helpers


In [1]:
import re
import sys
from pathlib import Path
from collections import Counter
from fractions import Fraction

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from symbolica import Expression, S

from model import (
    COLOR_ADJ_INDEX,
    COLOR_FUND_INDEX,
    LORENTZ_INDEX,
    SPINOR_INDEX,
    WEAK_ADJ_INDEX,
    WEAK_FUND_INDEX,
    CovD,
    Field,
    FieldStrength,
    Gamma,
    GaugeGroup,
    GaugeRepresentation,
    Model,
    StructureConstant,
    T,
    dirac_field,
)
from symbolic.spenso_structures import (
    gamma_matrix,
    gauge_generator,
    lorentz_levi_civita,
    structure_constant,
    weak_gauge_generator,
    weak_structure_constant,
)
from symbolic.vertex_engine import I

ANSI_ESCAPE_RE = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")


def clean(text):
    return ANSI_ESCAPE_RE.sub("", str(text))


def show(title, result):
    print("==========")
    print(title)
    if isinstance(result, dict):
        print(f"{len(result)} vertex signature(s)")
        print()
        for signature, expression in result.items():
            print("Vertex:", signature)
            print("Rule:", clean(expression))
            print()
    else:
        print(clean(result))
        print()


def show_model(model, *fields, compact_form=None, sum_notation=None):
    source_terms = model.lagrangian_decl.source_terms
    if source_terms:
        lagrangian_source = sum(source_terms[1:], source_terms[0]) if len(source_terms) > 1 else source_terms[0]
        show("Lagrangian", lagrangian_source)
    lagrangian = model.lagrangian()
    if fields:
        show("Feynman Rule", lagrangian.feynman_rule(*fields, include_delta=False))
    else:
        show("Feynman Rules", lagrangian.feynman_rule(include_delta=False))
    if compact_form is not None:
        show("Compact Form", compact_form)
    if sum_notation is not None:
        show("Sum Notation", sum_notation)



def show_expansion(title, model):
    """Show how a FieldStrength operator distributes into local interaction terms.

    For higher F^n operators the full set of Feynman rules can involve very
    high-leg vertices, so this helper reports the declared Lagrangian together
    with the number of compiled local terms and their per-vertex multiplicity.
    """
    source_terms = model.lagrangian_decl.source_terms
    lagrangian_source = sum(source_terms[1:], source_terms[0]) if len(source_terms) > 1 else source_terms[0]
    compiled = model.lagrangian()
    counts = Counter(len(term.fields) for term in compiled.terms)
    print("==========")
    print(title)
    print("Lagrangian:", clean(lagrangian_source))
    print(f"Compiled into {len(compiled.terms)} local interaction term(s)")
    for n_fields in sorted(counts):
        print(f"  {n_fields}-point pieces: {counts[n_fields]}")
    print()


### Symbols, fields, and gauge groups

The same `U1QED`, `SU3C`, and `SU2L` declarations used in `list_lagrangians.ipynb`.

In [2]:
mu, nu, rho, sigma = S("mu"), S("nu"), S("rho"), S("sigma")
a, b, c = S("a"), S("b"), S("c")
i, j = S("i"), S("j")
s1, s2, s3 = S("s1"), S("s2"), S("s3")
aC, aW = S("aC"), S("aW")
eQED, gS, g2 = S("eQED"), S("gS"), S("g2")

Photon = Field("A", spin=1, self_conjugate=True, symbol=S("A0"), indices=(LORENTZ_INDEX,))
Gluon = Field("G", spin=1, self_conjugate=True, symbol=S("G0"), indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX))
W = Field("W", spin=1, self_conjugate=True, symbol=S("W0"), indices=(LORENTZ_INDEX, WEAK_ADJ_INDEX))
Psi = dirac_field("Psi", symbol=S("psi"), conjugate_symbol=S("psibar"))

COLOR_FUND_REP = GaugeRepresentation(index=COLOR_FUND_INDEX, generator_builder=gauge_generator, name="fund")
WEAK_DOUBLET_REP = GaugeRepresentation(index=WEAK_FUND_INDEX, generator_builder=weak_gauge_generator, name="doublet")

U1QED = GaugeGroup(name="U1QED", abelian=True, coupling=eQED, gauge_boson=Photon, charge="Q")
SU3C = GaugeGroup(
    name="SU3C",
    abelian=False,
    coupling=gS,
    gauge_boson=Gluon,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP,),
)
SU2L = GaugeGroup(
    name="SU2L",
    abelian=False,
    coupling=g2,
    gauge_boson=W,
    structure_constant=weak_structure_constant,
    representations=(WEAK_DOUBLET_REP,),
)

(U1QED, SU3C, SU2L)


(GaugeGroup(name='U1QED', abelian=True, coupling=eQED, gauge_boson=Field(name='A', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', dimension=None, is_flavor=False, prefix='mu'),), kind='vector', statistics='boson', symbol=A0, conjugate_symbol=None, mass=None, quantum_numbers={}, ghost_of=None, flavor_index=None, class_members=()), ghost_field=None, structure_constant=None, representations=(), charge='Q'),
 GaugeGroup(name='SU3C', abelian=False, coupling=gS, gauge_boson=Field(name='G', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', dimension=None, is_flavor=False, prefix='mu'), IndexType(name='ColorAdj', representation=Representation { rep: SelfDual(2), dim: Concrete(8) }, kind='color_adj', dimension=None, is_flavor=False, prefix='a')), kind='vector', statistics='boson', symbol

## Abelian field strength: photon kinetic term

An abelian `FieldStrength(U1, mu, nu)` takes **no** adjoint index.

In [3]:
photon_kinetic_model = Model(
    -(Expression.num(1) / Expression.num(4)) * FieldStrength(U1QED, mu, nu) * FieldStrength(U1QED, mu, nu),
)
show_expansion("Abelian F^2 (photon)", photon_kinetic_model)
show_model(photon_kinetic_model)


Abelian F^2 (photon)
Lagrangian: -1/4 * FieldStrength(U1QED, mu, nu) * FieldStrength(U1QED, mu, nu)
Compiled into 4 local interaction term(s)
  2-point pieces: 4

Lagrangian
-1/4 * FieldStrength(U1QED, mu, nu) * FieldStrength(U1QED, mu, nu)

Feynman Rules
1 vertex signature(s)

Vertex: ('A', 'A')
Rule: -1𝑖*pcomp(q1,mu2)*pcomp(q2,mu1)+1𝑖*g(mink(4, mu1),mink(4, mu2))*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)



##  `epsilon(mu,nu,rho,sigma) F(mu,nu) F(rho,sigma)`


In [4]:
epsFF_u1 = Model(
    S("theta")
    * lorentz_levi_civita(mu, nu, rho, sigma)
    * FieldStrength(U1QED, mu, nu) * FieldStrength(U1QED, rho, sigma),
)
show_expansion("epsilon F F", epsFF_u1)
show_model(epsFF_u1)


epsilon F F
Lagrangian: theta*lor_levi_civita(mink(4, mu),mink(4, nu),mink(4, rho),mink(4, sigma)) * FieldStrength(U1QED, mu, nu) * FieldStrength(U1QED, rho, sigma)
Compiled into 4 local interaction term(s)
  2-point pieces: 4

Lagrangian
theta*lor_levi_civita(mink(4, mu),mink(4, nu),mink(4, rho),mink(4, sigma)) * FieldStrength(U1QED, mu, nu) * FieldStrength(U1QED, rho, sigma)

Feynman Rules
1 vertex signature(s)

Vertex: ('A', 'A')
Rule: -1𝑖*theta*lor_levi_civita(mink(4, mu1),mink(4, mu1_int),mink(4, mu2),mink(4, mu2_int))*pcomp(q1,mu1_int)*pcomp(q2,mu2_int)+1𝑖*theta*lor_levi_civita(mink(4, mu1),mink(4, mu1_int),mink(4, mu2_int),mink(4, mu2))*pcomp(q1,mu1_int)*pcomp(q2,mu2_int)-1𝑖*theta*lor_levi_civita(mink(4, mu2),mink(4, mu1_int),mink(4, mu1),mink(4, mu2_int))*pcomp(q1,mu2_int)*pcomp(q2,mu1_int)+1𝑖*theta*lor_levi_civita(mink(4, mu2),mink(4, mu1_int),mink(4, mu2_int),mink(4, mu1))*pcomp(q1,mu2_int)*pcomp(q2,mu1_int)+1𝑖*theta*lor_levi_civita(mink(4, mu1_int),mink(4, mu1),mink(4, mu2),

## Non-abelian field strength

A non-abelian `FieldStrength(SU3C, mu, nu, a)` **requires** an explicit adjoint index `a`.


In [5]:
gluon_kinetic_model = Model(
    -(Expression.num(1) / Expression.num(4)) * FieldStrength(SU3C, mu, nu, aC) * FieldStrength(SU3C, mu, nu, aC),
)
show_expansion("Non-abelian F^2 (gluon)", gluon_kinetic_model)
show_model(gluon_kinetic_model)

Non-abelian F^2 (gluon)
Lagrangian: -1/4 * FieldStrength(SU3C, mu, nu, aC) * FieldStrength(SU3C, mu, nu, aC)
Compiled into 9 local interaction term(s)
  2-point pieces: 4
  3-point pieces: 4
  4-point pieces: 1

Lagrangian
-1/4 * FieldStrength(SU3C, mu, nu, aC) * FieldStrength(SU3C, mu, nu, aC)

Feynman Rules
3 vertex signature(s)

Vertex: ('G', 'G')
Rule: 1𝑖*g(mink(4, mu1),mink(4, mu2))*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)-1𝑖*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu2)*pcomp(q2,mu1)

Vertex: ('G', 'G', 'G')
Rule: gS*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q1,mu3)-gS*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q2,mu3)-gS*g(mink(4, mu1),mink(4, mu3))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q1,mu2)+gS*g(mink(4, mu1),mink(4, mu3))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q3,mu2)+gS*g(mink(4, mu2),mink(4, mu3))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q2,mu1)-gS*g(mink(4, mu2),mink(4, mu3))*

## Cubic color operator: `f^{abc} F^a F^b F^c`

In [6]:
f_f3_model = Model(
    S("c3")
    * StructureConstant(a, b, c)
    * FieldStrength(SU3C, mu, nu, a)
    * FieldStrength(SU3C, nu, rho, b)
    * FieldStrength(SU3C, rho, mu, c),
)
show_expansion("f^{abc} F^a F^b F^c", f_f3_model)

show_model(f_f3_model)


f^{abc} F^a F^b F^c
Lagrangian: c3 * StructureConstant(a, b, c) * FieldStrength(SU3C, mu, nu, a) * FieldStrength(SU3C, nu, rho, b) * FieldStrength(SU3C, rho, mu, c)
Compiled into 27 local interaction term(s)
  3-point pieces: 8
  4-point pieces: 12
  5-point pieces: 6
  6-point pieces: 1

Lagrangian
c3 * StructureConstant(a, b, c) * FieldStrength(SU3C, mu, nu, a) * FieldStrength(SU3C, nu, rho, b) * FieldStrength(SU3C, rho, mu, c)

Feynman Rules
4 vertex signature(s)

Vertex: ('G', 'G', 'G')
Rule: 6*c3*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q1,mu2)*pcomp(q2,mu3)*pcomp(q3,mu1)-6*c3*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q1,mu3)*pcomp(q2,mu1)*pcomp(q3,mu2)-4*c3*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q1,mu1_int)*pcomp(q2,mu3)*pcomp(q3,mu1_int)-2*c3*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q1,mu2_int)*pcomp(q2,mu3)*pcomp(q3,mu2_int)+4*c3*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q

## Quartic operator: `(F^a F^a)(F^b F^b)`

`F^4` 3^4 = 81 local monomials

In [18]:
f4_model = Model(
    S("lam")
    * FieldStrength(SU3C, mu, nu, a) * FieldStrength(SU3C, mu, nu, a)
    * FieldStrength(SU3C, rho, sigma, b) * FieldStrength(SU3C, rho, sigma, b),
)
show_expansion("(F^a F^a)(F^b F^b)", f4_model)
show_model(f4_model)


(F^a F^a)(F^b F^b)
Lagrangian: lam * FieldStrength(SU3C, mu, nu, a) * FieldStrength(SU3C, mu, nu, a) * FieldStrength(SU3C, rho, sigma, b) * FieldStrength(SU3C, rho, sigma, b)
Compiled into 81 local interaction term(s)
  4-point pieces: 16
  5-point pieces: 32
  6-point pieces: 24
  7-point pieces: 8
  8-point pieces: 1

Lagrangian
lam * FieldStrength(SU3C, mu, nu, a) * FieldStrength(SU3C, mu, nu, a) * FieldStrength(SU3C, rho, sigma, b) * FieldStrength(SU3C, rho, sigma, b)

Feynman Rules
5 vertex signature(s)

Vertex: ('G', 'G', 'G', 'G')
Rule: 16𝑖*lam*g(mink(4, mu1),mink(4, mu2))*g(coad(8, a1),coad(8, a2))*g(mink(4, mu3),mink(4, mu4))*g(coad(8, a3),coad(8, a4))*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)*pcomp(q3,mu2_int)*pcomp(q4,mu2_int)+16𝑖*lam*g(mink(4, mu1),mink(4, mu2))*g(coad(8, a1),coad(8, a2))*g(mink(4, mu3),mink(4, mu4))*g(coad(8, a3),coad(8, a4))*pcomp(q1,mu2_int)*pcomp(q2,mu2_int)*pcomp(q3,mu1_int)*pcomp(q4,mu1_int)-16𝑖*lam*g(mink(4, mu1),mink(4, mu2))*g(coad(8, a1),coad(8, a2))*g(

## Mixed gauge groups: `F_{SU3}^2 * F_{SU2}^2`


In [8]:
mixed_model = Model(
    S("kappa")
    * FieldStrength(SU3C, mu, nu, aC) * FieldStrength(SU3C, mu, nu, aC)
    * FieldStrength(SU2L, rho, sigma, aW) * FieldStrength(SU2L, rho, sigma, aW),
    gauge_groups=(SU3C, SU2L),
    fields=(Gluon, W),
)
show_expansion("F_{SU3}^2 * F_{SU2}^2", mixed_model)
show_model(mixed_model)


F_{SU3}^2 * F_{SU2}^2
Lagrangian: kappa * FieldStrength(SU3C, mu, nu, aC) * FieldStrength(SU3C, mu, nu, aC) * FieldStrength(SU2L, rho, sigma, aW) * FieldStrength(SU2L, rho, sigma, aW)
Compiled into 81 local interaction term(s)
  4-point pieces: 16
  5-point pieces: 32
  6-point pieces: 24
  7-point pieces: 8
  8-point pieces: 1

Lagrangian
kappa * FieldStrength(SU3C, mu, nu, aC) * FieldStrength(SU3C, mu, nu, aC) * FieldStrength(SU2L, rho, sigma, aW) * FieldStrength(SU2L, rho, sigma, aW)

Feynman Rules
9 vertex signature(s)

Vertex: ('G', 'G', 'W', 'W')
Rule: 16𝑖*kappa*g(mink(4, mu1),mink(4, mu2))*g(coad(8, a1),coad(8, a2))*g(mink(4, mu3),mink(4, mu4))*g(coad(3, aw3),coad(3, aw4))*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)*pcomp(q3,mu2_int)*pcomp(q4,mu2_int)-16𝑖*kappa*g(mink(4, mu1),mink(4, mu2))*g(coad(8, a1),coad(8, a2))*g(coad(3, aw3),coad(3, aw4))*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)*pcomp(q3,mu4)*pcomp(q4,mu3)-16𝑖*kappa*g(coad(8, a1),coad(8, a2))*g(mink(4, mu3),mink(4, mu4))*g(coad(3, aw3)

### eps F F

In [19]:
epsFF_su3 = Model(
    lorentz_levi_civita(mu, nu, rho, sigma)
    * FieldStrength(SU3C, mu, nu,a) * FieldStrength(SU3C, rho, sigma,a),
)
show_expansion("epsilon F F", epsFF_su3)
show_model(epsFF_su3)


epsilon F F
Lagrangian: lor_levi_civita(mink(4, mu),mink(4, nu),mink(4, rho),mink(4, sigma)) * FieldStrength(SU3C, mu, nu, a) * FieldStrength(SU3C, rho, sigma, a)
Compiled into 9 local interaction term(s)
  2-point pieces: 4
  3-point pieces: 4
  4-point pieces: 1

Lagrangian
lor_levi_civita(mink(4, mu),mink(4, nu),mink(4, rho),mink(4, sigma)) * FieldStrength(SU3C, mu, nu, a) * FieldStrength(SU3C, rho, sigma, a)

Feynman Rules
3 vertex signature(s)

Vertex: ('G', 'G')
Rule: -1𝑖*g(coad(8, a1),coad(8, a2))*lor_levi_civita(mink(4, mu1),mink(4, mu1_int),mink(4, mu2),mink(4, mu2_int))*pcomp(q1,mu1_int)*pcomp(q2,mu2_int)+1𝑖*g(coad(8, a1),coad(8, a2))*lor_levi_civita(mink(4, mu1),mink(4, mu1_int),mink(4, mu2_int),mink(4, mu2))*pcomp(q1,mu1_int)*pcomp(q2,mu2_int)-1𝑖*g(coad(8, a1),coad(8, a2))*lor_levi_civita(mink(4, mu2),mink(4, mu1_int),mink(4, mu1),mink(4, mu2_int))*pcomp(q1,mu2_int)*pcomp(q2,mu1_int)+1𝑖*g(coad(8, a1),coad(8, a2))*lor_levi_civita(mink(4, mu2),mink(4, mu1_int),mink(4, mu2_int

## U1--`F(mu,nu) * psibar * Gamma(mu) * Gamma(nu) * psi`

In [20]:
fs_gamma_gamma_psi = Model(
    FieldStrength(U1QED, mu, nu)
    * Psi.bar * Gamma(mu) * Gamma(nu) * Psi,
)
show_expansion("F psibar Gamma(mu) Gamma(nu) psi", fs_gamma_gamma_psi)
show_model(fs_gamma_gamma_psi)


F psibar Gamma(mu) Gamma(nu) psi
Lagrangian: FieldStrength(U1QED, mu, nu) * Psi.bar * Gamma(mu) * Gamma(nu) * Psi
Compiled into 2 local interaction term(s)
  3-point pieces: 2

Lagrangian
FieldStrength(U1QED, mu, nu) * Psi.bar * Gamma(mu) * Gamma(nu) * Psi

Feynman Rules
1 vertex signature(s)

Vertex: ('Psi.bar', 'Psi', 'A')
Rule: -gamma(bis(4, spinor_decl_3),bis(4, i2),mink(4, mu1_int))*gamma(bis(4, i1),bis(4, spinor_decl_3),mink(4, mu3))*pcomp(q3,mu1_int)+gamma(bis(4, spinor_decl_3),bis(4, i2),mink(4, mu3))*gamma(bis(4, i1),bis(4, spinor_decl_3),mink(4, mu1_int))*pcomp(q3,mu1_int)



## `F^a(mu,nu) * psibar * Gamma(mu) * Gamma(nu) * t^a * psi` for `SU3C`



In [21]:
PsiColor = Field(
    "PsiC",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("psiC"),
    conjugate_symbol=S("psibarC"),
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX),
)

psi_bar_color = PsiColor.bar(index_labels={SPINOR_INDEX.kind: s1, COLOR_FUND_INDEX.kind: i})
psi_color = PsiColor(index_labels={SPINOR_INDEX.kind: s3, COLOR_FUND_INDEX.kind: j})

fs_gamma_gamma_q = Model(
    FieldStrength(SU3C, mu, nu, a)
    * psi_bar_color
    * Gamma(mu)
    * Gamma(nu)
    * T(a)
    * psi_color,
)
show_expansion("working declarative SU3 version", fs_gamma_gamma_q)
show_model(fs_gamma_gamma_q)

working declarative SU3 version
Lagrangian: FieldStrength(SU3C, mu, nu, a) * PsiC.bar * Gamma(mu) * Gamma(nu) * T(a) * PsiC
Compiled into 3 local interaction term(s)
  3-point pieces: 2
  4-point pieces: 1

Lagrangian
FieldStrength(SU3C, mu, nu, a) * PsiC.bar * Gamma(mu) * Gamma(nu) * T(a) * PsiC

Feynman Rules
2 vertex signature(s)

Vertex: ('PsiC.bar', 'PsiC', 'G')
Rule: -gamma(bis(4, spinor_decl_1),bis(4, i2),mink(4, mu1_int))*gamma(bis(4, i1),bis(4, spinor_decl_1),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))*pcomp(q3,mu1_int)+gamma(bis(4, spinor_decl_1),bis(4, i2),mink(4, mu3))*gamma(bis(4, i1),bis(4, spinor_decl_1),mink(4, mu1_int))*t(coad(8, a3),cof(3, c1),cof(3, c2))*pcomp(q3,mu1_int)

Vertex: ('PsiC.bar', 'PsiC', 'G', 'G')
Rule: 1𝑖*gS*gamma(bis(4, spinor_decl_1),bis(4, i2),mink(4, mu3))*gamma(bis(4, i1),bis(4, spinor_decl_1),mink(4, mu4))*t(coad(8, a),cof(3, c1),cof(3, c2))*f(coad(8, a),coad(8, a4),coad(8, a3))+1𝑖*gS*gamma(bis(4, i1),bis(4, spinor_decl_1),mink(4, mu3))*ga

### Raw `gamma(...) * t(...)` version, with **no** slot labels


In [13]:
fs_gamma_gamma_q_raw = Model(
    FieldStrength(SU3C, mu, nu, a)
    * PsiColor.bar
    * gamma_matrix(s1, s2, mu)
    * gamma_matrix(s2, s3, nu)
    * gauge_generator(a, i, j)
    * PsiColor,
)
show_expansion("raw gamma(...) * t(...) SU3 version (no slot labels)", fs_gamma_gamma_q_raw)
show_model(fs_gamma_gamma_q_raw)


raw gamma(...) * t(...) SU3 version (no slot labels)
Lagrangian: gamma(bis(4, s1),bis(4, s2),mink(4, mu))*gamma(bis(4, s2),bis(4, s3),mink(4, nu))*t(coad(8, a),cof(3, i),cof(3, j)) * FieldStrength(SU3C, mu, nu, a) * PsiC.bar * PsiC
Compiled into 3 local interaction term(s)
  3-point pieces: 2
  4-point pieces: 1

Lagrangian
gamma(bis(4, s1),bis(4, s2),mink(4, mu))*gamma(bis(4, s2),bis(4, s3),mink(4, nu))*t(coad(8, a),cof(3, i),cof(3, j)) * FieldStrength(SU3C, mu, nu, a) * PsiC.bar * PsiC

Feynman Rules
2 vertex signature(s)

Vertex: ('PsiC.bar', 'PsiC', 'G')
Rule: -gamma(bis(4, s2),bis(4, i2),mink(4, mu1_int))*gamma(bis(4, i1),bis(4, s2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))*pcomp(q3,mu1_int)+gamma(bis(4, s2),bis(4, i2),mink(4, mu3))*gamma(bis(4, i1),bis(4, s2),mink(4, mu1_int))*t(coad(8, a3),cof(3, c1),cof(3, c2))*pcomp(q3,mu1_int)

Vertex: ('PsiC.bar', 'PsiC', 'G', 'G')
Rule: 1𝑖*gS*gamma(bis(4, s2),bis(4, i2),mink(4, mu3))*gamma(bis(4, i1),bis(4, s2),mink(4, mu4))*t(coad

## Weak `f^{abc} F^a F^b F^c` for `SU2L` (raw `weak_structure_constant`)

The adjoint representation is resolved from the gauge group, not assumed to be
color. A raw `weak_structure_constant(...)` contracted with `FieldStrength(SU2L, ...)`
therefore lowers entirely in the SU(2) **weak** adjoint (`coad(3)`), with no
color `coad(8)` leaking in. This is the same operator shape as the SU(3) cubic
color invariant, but the structure constant and field strengths carry weak
indices end to end.

In [14]:
aw1, aw2, aw3 = S("aw1"), S("aw2"), S("aw3")
mw, nw, rw = S("mw"), S("nw"), S("rw")

weak_f_cubed = Model(
    weak_structure_constant(aw1, aw2, aw3)
    * FieldStrength(SU2L, mw, nw, aw1)
    * FieldStrength(SU2L, nw, rw, aw2)
    * FieldStrength(SU2L, rw, mw, aw3),
    name="weak-fF3",
    gauge_groups=(SU2L,),
)
show_expansion("SU(2) f^{abc} F^a F^b F^c (weak adjoint)", weak_f_cubed)

# Confirm every adjoint slot is the weak coad(3), never the color coad(8).
weak_couplings = "".join(clean(term.coupling) for term in weak_f_cubed.lagrangian().terms)
print("color coad(8) occurrences (expect 0):", weak_couplings.count("coad(8"))
print("weak  coad(3) occurrences (expect > 0):", weak_couplings.count("coad(3"))
show_model(weak_f_cubed, W, W, W)

SU(2) f^{abc} F^a F^b F^c (weak adjoint)
Lagrangian: f(coad(3, aw1),coad(3, aw2),coad(3, aw3)) * FieldStrength(SU2L, mw, nw, aw1) * FieldStrength(SU2L, nw, rw, aw2) * FieldStrength(SU2L, rw, mw, aw3)
Compiled into 27 local interaction term(s)
  3-point pieces: 8
  4-point pieces: 12
  5-point pieces: 6
  6-point pieces: 1

color coad(8) occurrences (expect 0): 0
weak  coad(3) occurrences (expect > 0): 162
Lagrangian
f(coad(3, aw1),coad(3, aw2),coad(3, aw3)) * FieldStrength(SU2L, mw, nw, aw1) * FieldStrength(SU2L, nw, rw, aw2) * FieldStrength(SU2L, rw, mw, aw3)

Feynman Rule
f(coad(3, aw3),coad(3, aw1),coad(3, aw2))*pcomp(q1,mu2)*pcomp(q2,mu3)*pcomp(q3,mu1)-f(coad(3, aw3),coad(3, aw1),coad(3, aw2))*pcomp(q1,mu3)*pcomp(q2,mu1)*pcomp(q3,mu2)-g(mink(4, mu1),mink(4, mu2))*f(coad(3, aw3),coad(3, aw1),coad(3, aw2))*pcomp(q1,mu1_int)*pcomp(q2,mu3)*pcomp(q3,mu1_int)+g(mink(4, mu1),mink(4, mu2))*f(coad(3, aw3),coad(3, aw1),coad(3, aw2))*pcomp(q1,mu3)*pcomp(q2,mu2_int)*pcomp(q3,mu2_int)-g(mink(4,

## Mixed `SU3C` / `SU2L` raw tensors in one operator

The unified pipeline routes each index to the slot that matches its
representation, even when several gauge groups appear at once and the tensors
are raw. Here a colored quark line carries a raw color generator
`gauge_generator(aC, ...)` contracted with `FieldStrength(SU3C, ...)`, while a
weak-doublet lepton line carries a raw `weak_gauge_generator(aW, ...)`
contracted with `FieldStrength(SU2L, ...)`. No slot labels are written on the
fermions: the color generator claims the `cof(3)` quark slots, the weak
generator claims the `cof(2)` doublet slots, and the two adjoint lines never
cross.

In [15]:
QuarkC = dirac_field("qC", indices=(COLOR_FUND_INDEX,), symbol=S("qC"), conjugate_symbol=S("qCbar"))
LeptonW = dirac_field("lW", indices=(WEAK_FUND_INDEX,), symbol=S("lW"), conjugate_symbol=S("lWbar"))

mC, nC, mW2, nW2 = S("mC"), S("nC"), S("mW2"), S("nW2")
scA, scB, scC = S("scA"), S("scB"), S("scC")
icC, jcC = S("icC"), S("jcC")
swA, swB, swC = S("swA"), S("swB"), S("swC")
iwW, jwW = S("iwW"), S("jwW")

mixed_raw = Model(
    FieldStrength(SU3C, mC, nC, aC)
    * QuarkC.bar
    * gamma_matrix(scA, scB, mC)
    * gamma_matrix(scB, scC, nC)
    * gauge_generator(aC, icC, jcC)
    * QuarkC
    * FieldStrength(SU2L, mW2, nW2, aW)
    * LeptonW.bar
    * gamma_matrix(swA, swB, mW2)
    * gamma_matrix(swB, swC, nW2)
    * weak_gauge_generator(aW, iwW, jwW)
    * LeptonW,
    name="mixed-raw",
    gauge_groups=(SU3C, SU2L),
    fields=(QuarkC, LeptonW, Gluon, W),
)
show_expansion("mixed SU(3)/SU(2) raw current x current", mixed_raw)

# The distinct generator tensors that appear: color t carries coad(8)+cof(3),
# weak t carries coad(3)+cof(2), and the two never mix.
generators = set()
for term in mixed_raw.lagrangian().terms:
    for match in re.findall(r"t\(coad\(\d+,[^)]*\),cof\(\d+,[^)]*\),cof\(\d+,[^)]*\)\)", clean(term.coupling)):
        generators.add(re.sub(r",[^),]*", "", match))  # keep only the dimensions
for gen in sorted(generators):
    print(gen)

mixed SU(3)/SU(2) raw current x current
Lagrangian: gamma(bis(4, scA),bis(4, scB),mink(4, mC))*gamma(bis(4, scB),bis(4, scC),mink(4, nC))*gamma(bis(4, swA),bis(4, swB),mink(4, mW2))*gamma(bis(4, swB),bis(4, swC),mink(4, nW2))*t(coad(3, aW),cof(2, iwW),cof(2, jwW))*t(coad(8, aC),cof(3, icC),cof(3, jcC)) * FieldStrength(SU3C, mC, nC, aC) * qC.bar * qC * FieldStrength(SU2L, mW2, nW2, aW) * lW.bar * lW
Compiled into 9 local interaction term(s)
  6-point pieces: 4
  7-point pieces: 4
  8-point pieces: 1

t(coad(3))))
t(coad(8))))


## Strict-mode errors

- a non-abelian field strength **must** carry an adjoint index,
- an abelian field strength **must not**,
- every adjoint index must be contracted (no open color indices).

In [16]:
def show_error(title, build):
    print("==========")
    print(title)
    try:
        build()
        print("  (no error raised)")
    except ValueError as exc:
        print("  ValueError:", clean(exc))
    print()


show_error(
    "Non-abelian field strength without an adjoint index",
    lambda: Model(
        -(Expression.num(1) / Expression.num(4)) * FieldStrength(SU3C, mu, nu) * FieldStrength(SU3C, mu, nu),
    ).lagrangian(),
)

show_error(
    "Abelian field strength with an adjoint index",
    lambda: Model(
        -(Expression.num(1) / Expression.num(4)) * FieldStrength(U1QED, mu, nu, a) * FieldStrength(U1QED, mu, nu, a),
    ).lagrangian(),
)

show_error(
    "Open (uncontracted) adjoint color index",
    lambda: Model(
        S("c") * FieldStrength(SU3C, mu, nu, a) * FieldStrength(SU3C, rho, sigma, b),
    ).lagrangian(),
)


Non-abelian field strength without an adjoint index
  ValueError: FieldStrength compilation: non-abelian gauge group 'SU3C' field strength requires an explicit adjoint index, e.g. FieldStrength(group, mu, nu, a).

Abelian field strength with an adjoint index
  ValueError: FieldStrength compilation: abelian gauge group 'U1QED' field strength does not take an adjoint index; write FieldStrength(group, mu, nu).

Open (uncontracted) adjoint color index
  ValueError: FieldStrength monomial has open (uncontracted) adjoint index/indices ['python::{}::a', 'python::{}::b']; the Lagrangian term must be a gauge singlet. Contract them against another field strength or a tensor carrying the matching adjoint label(s).



In [17]:
import hashlib
import json

from symbolic.tensor_canonicalization import canonize_full


def regression_canon(expr):
    return canonize_full(
        expr,
        spinor_indices=(
            S("i1"),
            S("i2"),
            S("s1"),
            S("s2"),
            S("s3"),
            S("spinor_decl_1"),
            S("spinor_decl_2"),
            S("spinor_decl_3"),
        ),
        lorentz_indices=(
            S("mu1"),
            S("mu2"),
            S("mu3"),
            S("mu4"),
            S("mu5"),
            S("mu6"),
            S("mu7"),
            S("mu8"),
            S("mu1_int"),
            S("mu2_int"),
            S("mu3_int"),
            S("mu4_int"),
        ),
        adjoint_indices=(
            S("a"),
            S("a1"),
            S("a2"),
            S("a3"),
            S("a4"),
            S("a5"),
            S("a6"),
            S("a7"),
            S("a8"),
            S("aC"),
            S("aW"),
        ),
        color_fund_indices=(S("c1"), S("c2"), S("i"), S("j")),
        weak_fund_indices=(S("w1"), S("w2"), S("w3"), S("w4")),
        weak_adj_indices=(
            S("aw1"),
            S("aw2"),
            S("aw3"),
            S("aw4"),
            S("aw5"),
            S("aw6"),
            S("aw7"),
            S("aw8"),
        ),
        run_color=False,
    ).expand().to_canonical_string()


def regression_digest(expr):
    return hashlib.sha256(regression_canon(expr).encode()).hexdigest()


def term_signature(term):
    bindings = []
    for binding in term.index_bindings:
        bindings.append(
            (
                (binding.index.name, binding.index.kind),
                tuple(sorted((slot.occurrence, slot.slot) for slot in binding.field_slots)),
                tuple(sorted((deriv.target, deriv.ordinal) for deriv in binding.derivatives)),
            )
        )
    return {
        "fields": tuple(
            (
                occ.field.name,
                bool(occ.conjugated),
                tuple((index.name, index.kind) for index in occ.field.indices),
            )
            for occ in term.fields
        ),
        "derivatives": tuple(
            (
                action.target,
                regression_canon(action.lorentz_index)
                if hasattr(action.lorentz_index, "to_atom_tree")
                else str(action.lorentz_index),
            )
            for action in term.derivatives
        ),
        "closed_dirac_bilinears": tuple(term.closed_dirac_bilinears),
        "bindings": tuple(sorted(bindings)),
        "sector": term.sector,
        "origin": term.origin,
        "coupling_digest": regression_digest(term.coupling),
    }


def term_set_digest(model):
    payload = [term_signature(term) for term in model.lagrangian().terms]
    blob = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(blob.encode()).hexdigest()


expected_compiled_term_counts = {
    "photon_kinetic_model": 4,
    "photon_dual_density_model": 4,
    "gluon_kinetic_model": 9,
    "f_f3_model": 27,
    "f4_model": 81,
    "mixed_model": 81,
    "epsFF_su3": 9,
    "fs_gamma_gamma_psi": 2,
    "fs_gamma_gamma_q": 3,
    "fs_gamma_gamma_q_raw": 3,
}

got_compiled_term_counts = {
    "photon_kinetic_model": len(photon_kinetic_model.lagrangian().terms),
    "photon_dual_density_model": len(epsFF_u1.lagrangian().terms),
    "gluon_kinetic_model": len(gluon_kinetic_model.lagrangian().terms),
    "f_f3_model": len(f_f3_model.lagrangian().terms),
    "f4_model": len(f4_model.lagrangian().terms),
    "mixed_model": len(mixed_model.lagrangian().terms),
    "epsFF_su3": len(epsFF_su3.lagrangian().terms),
    "fs_gamma_gamma_psi": len(fs_gamma_gamma_psi.lagrangian().terms),
    "fs_gamma_gamma_q": len(fs_gamma_gamma_q.lagrangian().terms),
    "fs_gamma_gamma_q_raw": len(fs_gamma_gamma_q_raw.lagrangian().terms),
}

assert got_compiled_term_counts == expected_compiled_term_counts, got_compiled_term_counts

expected_term_set_digests = {
    "photon_kinetic_model": "4e3578256d153f97e6355335d777a6557964e8f4be4d130ebefa9f81c247a7c5",
    "photon_dual_density_model": "1ac0e305eff980f434d7455597ab1fe20e0b9352eeef829a13df31a7359301e9",
    "gluon_kinetic_model": "550a3107b5a8b0097f5e7097ad49ab1c63380cabff87bdc5a5f2ccd09e1c81e6",
    "f_f3_model": "bc75ddb4d6ca25648d9fb421c0e19a58925a67c4855ef19851b5488944c6b893",
    "f4_model": "b766e2f212dd5f09518a8af847fd8e1a0cad2e157bb1de5a02e2cf51b2ef042a",
    "mixed_model": "f0c5661f9fa82a10e6b32c4f8b2f66b5bcfe6e2a3f5f6eaae63fd05d1f32b5a6",
    "epsFF_su3": "1642292e54b3414ce119e9babc99d332251061bc7d0efc1b50fcfbb1d3ebcda5",
    "fs_gamma_gamma_psi": "23d538b4da7fe37f38d2fff017c74c283f988b9213e8b025e87b1d21831ccbf7",
    "fs_gamma_gamma_q": "199fb662798f0b67ac14fc656d49257baf32ecb3f3917e7882d3ad2df5280efb",
    "fs_gamma_gamma_q_raw": "199fb662798f0b67ac14fc656d49257baf32ecb3f3917e7882d3ad2df5280efb",
}

got_term_set_digests = {
    "photon_kinetic_model": term_set_digest(photon_kinetic_model),
    "photon_dual_density_model": term_set_digest(epsFF_u1),
    "gluon_kinetic_model": term_set_digest(gluon_kinetic_model),
    "f_f3_model": term_set_digest(f_f3_model),
    "f4_model": term_set_digest(f4_model),
    "mixed_model": term_set_digest(mixed_model),
    "epsFF_su3": term_set_digest(epsFF_su3),
    "fs_gamma_gamma_psi": term_set_digest(fs_gamma_gamma_psi),
    "fs_gamma_gamma_q": term_set_digest(fs_gamma_gamma_q),
    "fs_gamma_gamma_q_raw": term_set_digest(fs_gamma_gamma_q_raw),
}

assert got_term_set_digests == expected_term_set_digests, got_term_set_digests

expected_rule_digests = {
    "photon_vertex_rule": "aa49eedc478f9b1e5b8a58a10967cc026d9ccda527f651e695c64ef534cbff0d",
    "su3_qqg_rule": "bffbd3f888faeee5101ab99b71cb4b8102f4c74627ac032504f3138127d14f60",
    "su3_qqgg_rule": "29b6647a2c28a279f85e54be773743903fb90246f455dc68982568231050827f",
}

got_rule_digests = {
    "photon_vertex_rule": regression_digest(
        fs_gamma_gamma_psi.lagrangian().feynman_rule(Psi.bar, Psi, Photon, include_delta=False)
    ),
    "declared_su3_qqg_rule": regression_digest(
        fs_gamma_gamma_q.lagrangian().feynman_rule(PsiColor.bar, PsiColor, Gluon, include_delta=False)
    ),
    "raw_su3_qqg_rule": regression_digest(
        fs_gamma_gamma_q_raw.lagrangian().feynman_rule(PsiColor.bar, PsiColor, Gluon, include_delta=False)
    ),
    "declared_su3_qqgg_rule": regression_digest(
        fs_gamma_gamma_q.lagrangian().feynman_rule(PsiColor.bar, PsiColor, Gluon, Gluon, include_delta=False)
    ),
    "raw_su3_qqgg_rule": regression_digest(
        fs_gamma_gamma_q_raw.lagrangian().feynman_rule(PsiColor.bar, PsiColor, Gluon, Gluon, include_delta=False)
    ),
}

assert got_rule_digests["photon_vertex_rule"] == expected_rule_digests["photon_vertex_rule"], got_rule_digests["photon_vertex_rule"]
for key in ("declared_su3_qqg_rule", "raw_su3_qqg_rule"):
    assert got_rule_digests[key] == expected_rule_digests["su3_qqg_rule"], (key, got_rule_digests[key])
for key in ("declared_su3_qqgg_rule", "raw_su3_qqgg_rule"):
    assert got_rule_digests[key] == expected_rule_digests["su3_qqgg_rule"], (key, got_rule_digests[key])

print("Notebook regression checks passed.")


Notebook regression checks passed.
